# Plain／residual 對照

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/03-comparison/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：40步學習實驗

完成下面的完整案例後，另開code cell執行：

```python
!python scripts/run_learning_extensions.py --section 03-comparison
from IPython.display import SVG, display
display(SVG(filename='docs/assets/diagrams/03-comparison-learning.svg'))
```

資料、optimizer與結果的限制見網頁；這是獨立補充，不取代下方三步檢查。

In [1]:
"""Budget-matched small plain/residual comparison, not a ResNet benchmark."""
from time import perf_counter
import torch
from torch import nn


class Block(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.residual = residual
        self.branch = nn.Sequential(nn.Conv2d(4, 4, 3, padding=1, bias=False), nn.ReLU(),
                                    nn.Conv2d(4, 4, 3, padding=1, bias=False))

    def forward(self, x):
        correction = self.branch(x)
        return x + correction if self.residual else correction


class Classifier(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.stem = nn.Conv2d(3, 4, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(residual) for _ in range(3)])
        self.head = nn.Linear(4, 2)

    def forward(self, x):
        x = self.blocks(nn.functional.relu(self.stem(x)))
        return self.head(x.mean((2, 3)))


def data(n, shift):
    images, labels = torch.zeros(n, 3, 16, 16), torch.arange(n) % 2
    for i in range(n):
        left = 2 + (i + shift) % 4
        top = 3 + shift
        images[i, 0 if labels[i] == 0 else 2, top:top + 8, left:left + 8] = 1
    return images, labels


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    train_x, train_y = data(8, 0)
    val_x, val_y = data(4, 2)
    assert not any(torch.equal(t, v) for t in train_x for v in val_x)
    plain, residual = Classifier(False), Classifier(True)
    residual.load_state_dict(plain.state_dict())
    for p, r in zip(plain.parameters(), residual.parameters()):
        assert torch.equal(p, r)
    for name, model in (("plain", plain), ("residual", residual)):
        params = sum(p.numel() for p in model.parameters())
        assert params == 986
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
        before = model.stem.weight.detach().clone()
        start = perf_counter()
        model.train()
        for step in range(3):
            optimizer.zero_grad(set_to_none=True)
            logits = model(train_x)
            loss = nn.functional.cross_entropy(logits, train_y)
            loss.backward()
            grad_norm = model.stem.weight.grad.norm().item()
            assert grad_norm > 0 and torch.isfinite(loss)
            assert torch.isfinite(model.stem.weight.grad).all()
            optimizer.step()
            print(f"{name} step={step}, loss={loss.item():.4f}, stem_grad_norm={grad_norm:.6f}")
        elapsed = perf_counter() - start
        assert not torch.equal(before, model.stem.weight)
        model.eval()
        with torch.no_grad():
            acc = (model(val_x).argmax(1) == val_y).float().mean().item()
        print(f"{name}: params={params}, MACs/image=248840, shortcut_adds/image="
              f"{3072 if name == 'residual' else 0}, validation_accuracy={acc:.2f}, "
              f"3_step_seconds={elapsed:.4f}")
    print("Same initial weights/data/optimizer/steps; 3 steps and 4 validation images do not rank architectures.")


if __name__ == "__main__":
    main()


plain step=0, loss=0.6938, stem_grad_norm=0.000987
plain step=1, loss=0.6937, stem_grad_norm=0.001004
plain step=2, loss=0.6936, stem_grad_norm=0.001014
plain: params=986, MACs/image=248840, shortcut_adds/image=0, validation_accuracy=0.50, 3_step_seconds=0.0124
residual step=0, loss=0.6922, stem_grad_norm=0.146834
residual step=1, loss=0.6888, stem_grad_norm=0.147695
residual step=2, loss=0.6855, stem_grad_norm=0.146226
residual: params=986, MACs/image=248840, shortcut_adds/image=3072, validation_accuracy=0.50, 3_step_seconds=0.0081
Same initial weights/data/optimizer/steps; 3 steps and 4 validation images do not rank architectures.
